# 01 — ILPD Data Audit & Provenance Verification

This notebook verifies the original **Indian Liver Patient Dataset (ILPD)** structure, SHA-256 provenance, missing values, exact & near duplicates, domain consistency checks, and outlier profiles using reusable modules from `src/liver_disease/data/`.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import pandas as pd
import numpy as np
from IPython.display import display, Image
from liver_disease.data.loader import ensure_raw_dataset, get_dataset_provenance, load_raw_dataset
from liver_disease.data.validation import run_comprehensive_data_audit

raw_path = ensure_raw_dataset()
prov = get_dataset_provenance(raw_path)
df = load_raw_dataset(raw_path)
print('Dataset Provenance:')
for k, v in prov.items():
    print(f'  {k}: {v}')
display(df.head())

Dataset Provenance:
  filename: Indian Liver Patient Dataset (ILPD).csv
  path: E:\Ahmed\DataScience\PythonProjects\Liver Disease\Antigravity\data\raw\Indian Liver Patient Dataset (ILPD).csv
  size_bytes: 23817
  sha256: dbb065e6bf2b226b6d9c04f066ddc0b45173861dfb3a4df83f3710d3927ef5de
  source_name: UCI Machine Learning Repository — ILPD (Indian Liver Patient Dataset)
  source_url: https://archive.ics.uci.edu/dataset/225/ilpd+indian+liver+patient+dataset
  original_creators: Bendi Venkata Ramana, M. Surendra Prasad Babu, N. B. Venkateswarlu (2012)
  geographic_origin: North East of Andhra Pradesh, India


,Age,Gender,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio,Selector
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,1
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490,60,68,7.0,3.3,0.89,1
3,58,Male,1.0,0.4,182,14,20,6.8,3.4,1.00,1
4,72,Male,3.9,2.0,195,27,59,7.3,2.4,0.40,1


In [2]:
audit = run_comprehensive_data_audit(df)
print('Shape:', audit['shape'])
print('Target distribution:', audit['target_distribution'])
print('Missing values:', audit['missing_values'])
print('Exact duplicate extra rows:', audit['duplicates']['exact_duplicate_extra_rows'])
print('Target conflicts among feature duplicates:', len(audit['duplicates']['feature_duplicate_target_conflicts']))
print('Near-duplicate pairs (std L2 < 0.25):', len(audit['duplicates']['near_duplicate_pairs_z_lt_0_25']))
print('Domain consistency checks:', audit['domain_consistency_checks'])

Shape: {'rows': 583, 'columns': 11}
Target distribution: {'raw_counts': {'1': 416, '2': 167}, 'liver_disease_count': 416, 'liver_disease_pct': 71.36, 'no_liver_disease_count': 167, 'no_liver_disease_pct': 28.64, 'imbalance_ratio_pos_to_neg': 2.491}
Missing values: {'total_missing_cells': 4, 'rows_with_any_missing': 4, 'by_column': {'A/G Ratio': {'count': 4, 'percentage': 0.6861, 'row_indices': [209, 241, 253, 312]}}}
Exact duplicate extra rows: 13
Target conflicts among feature duplicates: 0
Near-duplicate pairs (std L2 < 0.25): 12
Domain consistency checks: {'direct_gt_total_bilirubin_count': 3, 'direct_gt_total_bilirubin_indices': [246, 261, 279], 'albumin_gt_total_proteins_count': 0, 'albumin_gt_total_proteins_indices': [], 'non_positive_values': {}, 'ag_ratio_diff_gt_0_25_count': 8, 'ag_ratio_diff_gt_0_25_indices': [33, 34, 267, 415, 484, 491, 548, 575]}


In [3]:
num_summary = pd.DataFrame(audit['numeric_summary']).T
display(num_summary[['count', 'missing', 'mean', 'std', 'min', 'median', 'max', 'skewness', 'iqr_outlier_count', 'iqr_outlier_pct']])

,count,missing,mean,std,min,median,max,skewness,iqr_outlier_count,iqr_outlier_pct
Age,583.0,0.0,44.7461,16.1898,4.0,45.00,90.0,-0.0294,0.0,0.00
TB,583.0,0.0,3.2988,6.2095,0.4,1.00,75.0,4.9075,84.0,14.41
DB,583.0,0.0,1.4861,2.8085,0.1,0.30,19.7,3.2124,81.0,13.89
Alkphos,583.0,0.0,290.5763,242.9380,63.0,208.00,2110.0,3.7651,69.0,11.84
Sgpt,583.0,0.0,80.7136,182.6204,10.0,35.00,2000.0,6.5492,73.0,12.52
Sgot,583.0,0.0,109.9108,288.9185,10.0,42.00,4929.0,10.5462,66.0,11.32
TP,583.0,0.0,6.4832,1.0855,2.7,6.60,9.6,-0.2857,8.0,1.37
ALB,583.0,0.0,3.1419,0.7955,0.9,3.10,5.5,-0.0437,0.0,0.00
A/G Ratio,579.0,4.0,0.9471,0.3196,0.3,0.93,2.8,0.9923,10.0,1.73


## Key Audit Decisions

1. **Missing values**: Exactly 4 missing entries in `A/G Ratio` (0.69%). Imputed strictly inside training folds via median imputation.
2. **Exact duplicates**: 13 exact duplicate rows (all consecutive adjacent double-entries, 0 target conflicts). Deduplicated (`583 -> 570`) before train/test splitting to prevent identical-patient leakage.
3. **Outliers & anomalies**: Extreme bilirubin/enzyme values (11–14% IQR outliers) represent genuine severe liver pathology and are preserved intact. The 3 rows with `DB > TB` are documented transparently without silent modification.